# Data-Efficient Vision Transformers for Small Image Datasets
### IT3071 Machine Learning — Explainer Notebook

This notebook walks through our full experiment end-to-end.

**The core question:** *Can a Vision Transformer (ViT) compete with a CNN when trained on a small dataset?*

**Short answer:** Not from scratch — but if we fine-tune a pretrained ViT, it wins easily.

**What we compare:**
| Model | Training | Expected result |
|---|---|---|
| ViT-Tiny (from scratch) | Random init, trained only on our small dataset | Worst — no inductive bias, no pretraining |
| ResNet-18 (CNN) | Random init | Mid — convolutional inductive bias helps |
| ViT-Tiny (pretrained) | ImageNet weights, fine-tuned | Best — pretraining compensates for small data |

**Why this ranking?** CNNs bake in two assumptions about images: *locality* (nearby pixels are related) and *translation invariance* (a cat is still a cat if you shift it left). ViTs make neither assumption — they learn these patterns from data. With enough data that's a strength (they can learn richer patterns). With a small dataset it's a weakness.

Run the cells top-to-bottom. GPU recommended for the training cells.

## 1. Setup

Install dependencies and import all modules. The `sys.path` insert lets us import from `src/` whether this notebook is run from the repo root or from inside `notebooks/`.

In [ ]:
# On Google Colab: clone the repo first, then run this cell.
# !git clone https://github.com/Dippy2003/data-efficient-vit.git
# %cd data-efficient-vit
# !pip install -r requirements.txt

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "") or "..")

import random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Fix random seeds so results are reproducible across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from src.data      import get_dataloaders, CIFAR10_CLASSES
from src.models    import build_model, count_parameters, MODEL_REGISTRY
from src.train     import get_device, train_model, load_checkpoint, DEV_CONFIG, FINAL_CONFIG
from src.evaluate  import (compute_accuracy, confusion_matrix_from_loader,
                           per_class_report, build_results_table, print_results_table)
from src.visualize import (plot_training_curves, plot_confusion_matrix,
                           plot_sample_predictions, plot_attention_overlay)

DEVICE = get_device()
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

---
## 2. What is a Vision Transformer?

A **Vision Transformer (ViT)** applies the Transformer architecture — originally designed for text — to images.

**How it works (step by step):**
1. **Patch embedding** — The image is split into a grid of fixed-size patches (16×16 pixels each). Each patch is flattened and linearly projected into a vector called a *token*, just like a word embedding in NLP.
2. **Positional encoding** — A learnable position embedding is added to each token so the model knows *where* each patch came from in the image.
3. **Transformer encoder** — The sequence of patch tokens is fed through stacked Multi-Head Self-Attention + Feed-Forward layers. Each token can attend to every other token equally, learning which patches are important to each other.
4. **Classification head** — A special `[CLS]` token is prepended; its final representation is passed through a linear layer to produce class logits.

**Why ViTs struggle on small datasets:**
- Self-attention is *global* — every patch attends to every other patch from the start. There is no built-in assumption that nearby patches are more related than distant ones.
- CNNs use *convolution kernels* that enforce locality by design: each neuron only looks at a small neighbourhood. This is a strong prior that turns out to be correct for most natural images.
- When training data is scarce, priors matter a lot. The CNN gets spatial priors for free; the ViT has to discover them from examples — and with too few examples, it never quite does.

> **Key takeaway:** The from-scratch ViT and the CNN have similar numbers of parameters (~5M vs ~11M), so the performance gap is about *architecture*, not *size*.

---
## 3. Data Pipeline

We use **CIFAR-10**: 60,000 32×32 colour images across 10 classes (50,000 train / 10,000 test). It's a classic benchmark that's small enough to train on consumer hardware, but large enough to show meaningful differences between models.

**Key pipeline choices:**
- Images are resized to **224×224** even though CIFAR-10 is natively 32×32. This is necessary because the pretrained ViT's position embeddings were trained at 224×224 resolution (matching ImageNet). Using a smaller size would break the positional encoding.
- **Augmentation** (random crop + horizontal flip) is applied only to the training set — validation/test use the clean image so metrics reflect real performance.
- `subset_fraction` controls how much training data to use. **0.05 during development** (fast), **1.0 for the final reported run**.

> Change `USE_FULL_DATA = False` → `True` before your final run.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
USE_FULL_DATA = False          # set True for the final graded run
IMAGE_SIZE    = 224
BATCH_SIZE    = 64
# ─────────────────────────────────────────────────────────────────────────────

config = FINAL_CONFIG if USE_FULL_DATA else DEV_CONFIG

loaders = get_dataloaders(
    dataset_name    = "cifar10",
    data_root       = "../data",
    image_size      = IMAGE_SIZE,
    batch_size      = BATCH_SIZE,
    subset_fraction = config["subset_fraction"],
    num_workers     = 0,          # set to 2 on Linux/Colab for faster loading
)

print(f"Classes : {CIFAR10_CLASSES}")
print(f"Train   : {len(loaders['train'].dataset)} samples")
print(f"Val     : {len(loaders['val'].dataset)} samples")
print(f"Test    : {len(loaders['test'].dataset)} samples")

### Sample images from CIFAR-10

Let's visualise one example per class to see what the model is working with. Note how similar some classes look at 32×32 (cat vs dog, automobile vs truck) — these are the pairs we expect to see confused in the confusion matrix later.

In [ ]:
CIFAR10_MEAN = np.array([0.4914, 0.4822, 0.4465])
CIFAR10_STD  = np.array([0.2470, 0.2435, 0.2616])

def denorm(tensor):
    """Undo the ImageNet-style normalisation so we can display the image."""
    img = tensor.numpy().transpose(1, 2, 0)
    return np.clip(img * CIFAR10_STD + CIFAR10_MEAN, 0, 1)

# Collect one example per class from the test loader (no augmentation)
seen = {}
for images, labels in loaders["test"]:
    for img, lbl in zip(images, labels):
        cls = lbl.item()
        if cls not in seen:
            seen[cls] = img
        if len(seen) == 10:
            break
    if len(seen) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(denorm(seen[i]))
    ax.set_title(CIFAR10_CLASSES[i], fontsize=11)
    ax.axis("off")

fig.suptitle("CIFAR-10 — one example per class", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Model Architectures

We build three models using `src/models.py`'s `build_model()` factory.

**ViT-Tiny (from scratch)** — `vit_scratch`
- Architecture: 12 transformer blocks, 3 attention heads, embedding dim 192, patch size 16×16
- Initialisation: random — must learn *everything* from our small dataset
- Why "tiny"? The smallest standard ViT variant, keeping training time reasonable

**ResNet-18 (CNN)** — `cnn`
- Architecture: 8 residual blocks with skip connections, convolutions throughout
- Initialisation: random — same as ViT-scratch, fair comparison
- Why ResNet-18? A well-understood, widely-used baseline that isn't too large

**ViT-Tiny (pretrained)** — `vit_pretrained`
- Same architecture as ViT-scratch
- Initialisation: **ImageNet-21k pretrained weights** (loaded via `timm`)
- Why pretrained helps: the model already knows edges, textures, object parts — fine-tuning just adapts these to our 10 classes

> Notice that ViT-scratch has **fewer parameters than ResNet-18**. Any accuracy gap between them is about inductive bias, not capacity.

In [ ]:
print(f"{'Model':<20} {'Total params':>15} {'Trainable':>15}")
print("-" * 52)
for name in MODEL_REGISTRY:
    m = build_model(name, num_classes=10, img_size=IMAGE_SIZE)
    p = count_parameters(m)
    print(f"{name:<20} {p['total']:>15,} {p['trainable']:>15,}")

---
## 5. Training

### Optimiser & learning rate

All three models use **AdamW** with a **cosine learning-rate schedule**:
- AdamW adapts the learning rate per parameter and adds weight decay — better than plain SGD for transformers.
- Cosine decay starts at the configured LR and smoothly drops toward 0 by the final epoch, helping the model settle into a good minimum rather than oscillating.

**Learning rates differ by model** (see `src/train.get_optimizer()`):

| Model | LR | Reason |
|---|---|---|
| `vit_scratch` | 1e-3 | Needs large steps — learning everything from zero |
| `cnn` | 1e-3 | Same — random init |
| `vit_pretrained` | **1e-4** | Fine-tuning — must *nudge* pretrained weights, not overwrite them |

### Speed note

Training the ViT models is noticeably slower than the CNN on the same hardware, because self-attention is O(n²) in the number of patches. On an RTX 3050 Laptop GPU, each epoch on the full CIFAR-10 training set takes roughly 2–4 minutes per model. **Use `USE_FULL_DATA = False` while exploring**, then switch to `True` for the final run.

> `DEV_CONFIG` = `subset_fraction=0.05`, `num_epochs=2` — fast, for testing the code  
> `FINAL_CONFIG` = `subset_fraction=1.0`, `num_epochs=15` — for reported results

In [ ]:
histories = {}

for model_name in MODEL_REGISTRY:
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")

    model = build_model(model_name, num_classes=10, img_size=IMAGE_SIZE).to(DEVICE)
    histories[model_name] = train_model(
        model_name, model, loaders, DEVICE,
        num_epochs=config["num_epochs"],
    )

print("\nAll models trained!")

---
## 6. Training Curves

The training curves are the most informative visual in the experiment — not just *final* accuracy, but *how each model learns*.

**What to look for:**
- `vit_pretrained` (green) **starts high and improves quickly** — it already knows useful features from ImageNet pretraining.
- `cnn` (blue) **climbs steadily** — convolution priors help it extract useful features even from limited data.
- `vit_scratch` (red) **climbs slowly or stalls** — without priors or pretraining, it struggles to learn spatial structure from a small dataset.
- **Val loss diverging from train loss** is a sign of overfitting — more likely in `vit_scratch` because it has no regularisation via inductive bias.

Solid lines = training set, dashed = validation set.

In [ ]:
curve_path = plot_training_curves(
    history_dir="../outputs/history",
    save_path="../outputs/figures/training_curves.png",
)

from IPython.display import Image as IPyImage
IPyImage(curve_path)

---
## 7. Evaluation

We evaluate on the **held-out test set** (10,000 images, never seen during training). We reload each model's *best checkpoint* (saved during training by `train_model()`) rather than using the last epoch's weights — this avoids accidentally reporting a slightly-worse final epoch when an earlier one peaked higher.

**Metrics:**
- **Accuracy** — fraction of test images correctly classified (top-1)
- **Macro F1** — average F1 score across all 10 classes, giving equal weight to each class regardless of size. Better than accuracy when some classes are harder than others (which they are — cat and dog tend to be harder).

In [ ]:
# Reload best checkpoints saved during training
trained_models = {}
for name in MODEL_REGISTRY:
    m = build_model(name, num_classes=10, img_size=IMAGE_SIZE).to(DEVICE)
    trained_models[name] = load_checkpoint(m, name, DEVICE, checkpoint_dir="../outputs/checkpoints")

# Build and print the comparison table
rows = build_results_table(trained_models, loaders["test"], DEVICE, CIFAR10_CLASSES)
print_results_table(rows, save_path="../outputs/results_table.txt")

### Per-class precision, recall, F1

The per-class report breaks down *where* each model succeeds and fails. 

- **Precision** = "of the samples predicted as this class, how many actually were?" — penalises false positives
- **Recall** = "of the actual samples of this class, how many did the model find?" — penalises false negatives  
- **F1** = harmonic mean of precision and recall — useful single-number per class

Look for classes with near-zero recall — the model is essentially ignoring those classes, predicting something else for almost every example.

In [ ]:
for name, model in trained_models.items():
    print(f"\n{'='*55}")
    print(f"  Per-class report: {name}")
    print(f"{'='*55}")
    print(per_class_report(model, loaders["test"], DEVICE, CIFAR10_CLASSES))

### Confusion Matrices

A confusion matrix is an N×N grid where entry (i, j) = fraction of true-class-i samples predicted as class j.

- **Diagonal** = correct predictions (want these high — darker blue)
- **Off-diagonal** = errors (want these low — lighter)

**What to look for:** pairs like (cat, dog) and (automobile, truck) tend to appear as bright off-diagonal spots because they look visually similar at 32×32 resolution. The pretrained ViT should have a much cleaner diagonal than the from-scratch ViT, because it has already learned to distinguish these pairs from ImageNet pretraining.

In [ ]:
from IPython.display import Image as IPyImage, display

for name, model in trained_models.items():
    cm, classes = confusion_matrix_from_loader(
        model, loaders["test"], DEVICE, CIFAR10_CLASSES
    )
    path = plot_confusion_matrix(
        cm, classes, name,
        save_path=f"../outputs/figures/{name}_confusion.png"
    )
    print(f"\n{name}")
    display(IPyImage(path))

### Sample Predictions

Looking at individual predictions makes the accuracy numbers concrete. **Green title = correct, red = wrong.**

This is often more compelling than any statistic for a video walkthrough — you can point to a specific image and say "the pretrained ViT gets this right because it learned what a cat looks like from ImageNet, but the from-scratch ViT hasn't seen enough cats to be confident."

In [ ]:
for name, model in trained_models.items():
    path = plot_sample_predictions(
        model, loaders["test"], DEVICE, CIFAR10_CLASSES,
        model_name=name, n_samples=16,
        save_path=f"../outputs/figures/{name}_predictions.png",
    )
    print(f"\n{name}")
    display(IPyImage(path))

---
## 8. Attention Maps

Attention maps let us **see what the ViT is looking at** when it classifies an image.

### How we extract them

Each transformer block has a Multi-Head Self-Attention layer. We register a *forward hook* on the last block's attention module — this intercepts the softmax attention weights during a forward pass without modifying the model.

The **`[CLS]` token** (prepended to the patch sequence) is what the model ultimately uses for classification. So we take CLS token's attention row — the vector of weights it assigns to each patch — and reshape it from a flat 196-dimensional vector into a 14×14 spatial grid (14×14 = 196 patches for a 224×224 image with 16×16 patches).

We then **average across all attention heads** and **upsample to 224×224** with bilinear interpolation, giving a heatmap we can overlay on the original image.

### What to look for

- A **well-trained pretrained ViT** typically shows attention concentrated on the object (e.g. the body of a cat, the fuselage of a plane) rather than the background.
- A **from-scratch ViT** trained on little data usually shows a much more diffuse, noisy pattern — it hasn't learned what matters yet.

This is another way to visualise the data-efficiency gap beyond accuracy numbers.

In [ ]:
# Pick a sample image from the test set for attention visualisation
sample_images, sample_labels = next(iter(loaders["test"]))
sample_img   = sample_images[:1].to(DEVICE)
sample_class = CIFAR10_CLASSES[sample_labels[0].item()]
print(f"Sample image class: {sample_class}")

# Plot attention for both ViT variants so we can compare trained vs untrained
for name in ["vit_pretrained", "vit_scratch"]:
    model = trained_models[name]
    model.eval()
    path = plot_attention_overlay(
        model, sample_img,
        class_name=sample_class,
        model_name=name,
        save_path=f"../outputs/figures/{name}_attention.png",
    )
    print(f"\n{name}")
    display(IPyImage(path))

---
## 9. Conclusion

### Summary of findings

| Model | Expected accuracy (full CIFAR-10, 15 epochs) | Why |
|---|---|---|
| `vit_scratch` | ~50–60% | Learns slowly without priors; limited by small dataset |
| `cnn` (ResNet-18) | ~75–80% | Convolutional inductive bias compensates for limited data |
| `vit_pretrained` | ~88–92% | Pretrained features + fine-tuning = best of both worlds |

> Note: With only 5% of CIFAR-10 (dev mode), accuracies are much lower and the ranking is the same but the gaps compress. The gap **widens with more data** for `vit_scratch` because it needs data to learn what the CNN gets for free from its architecture.

### Key takeaways

1. **ViTs are not data-efficient from scratch.** Self-attention is a powerful but unconstrained mechanism — it needs a lot of examples to discover spatial structure that convolutions encode by design.

2. **Pretraining changes everything.** A ViT pretrained on millions of images effectively "downloads" the inductive bias it needs from data rather than architecture. After pretraining, it consistently outperforms the CNN baseline.

3. **Parameter count ≠ performance.** ViT-Tiny (5.5M params) and ResNet-18 (11.2M params) have different accuracies from scratch — the gap is about *how* parameters are used, not how many there are.

4. **The gap is visible beyond accuracy.** Training curves, confusion matrices, per-class reports, and attention maps all reinforce the same story from different angles — which is exactly why we compute all four.

### Further experiments you could try

- Swap CIFAR-10 for a plant-disease or MedMNIST dataset — just add a branch in `src/data.get_dataset()` and nothing else changes.
- Try a smaller `subset_fraction` (0.01, 0.05, 0.1, 0.5, 1.0) and plot accuracy vs dataset size to show the data-efficiency curve explicitly.
- Add DeiT (Data-efficient Image Transformers), which uses knowledge distillation to approach CNN data-efficiency without pretraining — a natural extension.

---
## 10. How to run this notebook

### On Google Colab (recommended — free GPU)

1. Upload the whole repo folder to Google Drive, or clone with:
   ```
   !git clone https://github.com/Dippy2003/data-efficient-vit.git
   %cd data-efficient-vit
   ```
2. Open `notebooks/main.ipynb` in Colab
3. Go to **Runtime → Change runtime type → GPU** (T4 is fine and free)
4. Run the setup cell — it installs all dependencies
5. Set `USE_FULL_DATA = True` in the config cell before the final run

### Locally

```bash
pip install -r requirements.txt
jupyter notebook notebooks/main.ipynb
```

### Quick sanity check (before the full run)

```bash
python -m src.test_integration
```

Trains all 3 models for 1 epoch on 2% of CIFAR-10 and runs the full eval + visualize pipeline. Takes 2–5 minutes on CPU.

### Expected runtime on GPU (T4 Colab)

| Config | Data | Epochs | Per model | Total |
|---|---|---|---|---|
| DEV (`USE_FULL_DATA=False`) | 5% CIFAR-10 | 2 | ~2 min | ~6 min |
| FINAL (`USE_FULL_DATA=True`) | 100% CIFAR-10 | 15 | ~25–40 min | ~1.5–2 hr |

In [ ]:
# ── Final run helper ────────────────────────────────────────────────────────
# Run this cell (with USE_FULL_DATA = True above) to kick off the complete
# experiment in one go. All checkpoints, histories, and figures are saved
# automatically to outputs/ so you don't need to re-run training to
# regenerate plots.

if USE_FULL_DATA:
    print("Running FINAL config: full CIFAR-10, 15 epochs per model.")
    print("This will take ~1.5–2 hours on a GPU. Go get a coffee.")
else:
    print("Running DEV config: 5% data, 2 epochs. Fast, for testing.")
    print("Set USE_FULL_DATA = True in the config cell for the graded run.")

print(f"\nConfig in use:")
print(f"  subset_fraction : {config['subset_fraction']}")
print(f"  num_epochs      : {config['num_epochs']}")
print(f"  image_size      : {IMAGE_SIZE}")
print(f"  batch_size      : {BATCH_SIZE}")
print(f"  device          : {DEVICE}")

---
## 11. Extension: Data-efficiency curve (optional)

If you want to really nail the "data-efficiency" story, you can run each model
at multiple `subset_fraction` values and plot accuracy vs training-set size.

The expected shape of each curve:
- `vit_scratch` — stays low for small fractions, starts climbing only when it has enough data to learn spatial structure
- `cnn` — climbs quickly from the start (priors kick in immediately) then plateaus
- `vit_pretrained` — already high even at very small fractions (pretraining did the heavy lifting)

**The gap between `vit_scratch` and the other two widens as data decreases** —
that's the data-efficiency gap visualised directly.

```python
# Pseudocode — run this separately, not inside this notebook, as it multiplies training time
fractions = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]
results   = {name: [] for name in MODEL_REGISTRY}

for frac in fractions:
    loaders = get_dataloaders(subset_fraction=frac, ...)
    for name in MODEL_REGISTRY:
        model = build_model(name).to(device)
        train_model(name, model, loaders, device, num_epochs=10)
        model = load_checkpoint(model, name, device)
        acc   = compute_accuracy(model, loaders["test"], device)
        results[name].append(acc)

# Then plot results[name] vs fractions for each model
```

This produces the canonical "data-efficiency curve" referenced in the ViT paper
(Dosovitskiy et al., 2020), showing exactly when ViTs overtake CNNs as a
function of dataset size.

---
## References

1. Dosovitskiy, A. et al. (2020). *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale.* arXiv:2010.11929
   - Original ViT paper. Section 3.1 documents the data-efficiency gap: ViTs need JFT-300M to beat CNNs; ImageNet alone is not enough from scratch.

2. He, K. et al. (2016). *Deep Residual Learning for Image Recognition.* CVPR 2016.
   - ResNet paper. ResNet-18 is the CNN baseline used in this project.

3. Touvron, H. et al. (2021). *Training data-efficient image transformers & distillation through attention.* ICML 2021.
   - DeiT paper. Shows that knowledge distillation from a CNN teacher lets a ViT match CNN accuracy on ImageNet alone — the "data-efficient" extension mentioned in Section 11.

4. Loshchilov, I. & Hutter, F. (2019). *Decoupled Weight Decay Regularization.* ICLR 2019.
   - AdamW paper. The optimiser used for all three models in this project.

5. Ross Wightman (2019–). *timm: PyTorch Image Models.* github.com/rwightman/pytorch-image-models
   - Library used to load ViT-Tiny architecture and pretrained weights.